# 위성 스케줄링
**SIA Wevengers | 양안관계 OSINT-GEOINT 프로젝트**

- 반경: 30km
- 중복 좌표 제거 후 실행
- 구글 드라이브 연동 (세션 재시작 시 클론 불필요)

## 1. 구글 드라이브 연동 + GitHub 클론
**최초 1회만 실행** (이후 세션에서는 2번 셀부터 실행)

In [ ]:
from google.colab import drive
import os

# 드라이브 마운트
drive.mount('/content/drive')

REPO_PATH = '/content/drive/MyDrive/DS8_SIA_Project'
BRANCH = 'dashboard'

# 최초 1회: 클론
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/anna030608/DS8_SIA_Project.git {REPO_PATH}
    print('✅ 클론 완료')
else:
    print('✅ 이미 존재 - 클론 생략')

# 브랜치 전환 + 최신 내용 pull
%cd {REPO_PATH}
!git checkout {BRANCH}
!git pull origin {BRANCH}
print('✅ 준비 완료')

## 2. 세션 재시작 시 여기서부터 실행

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_PATH = '/content/drive/MyDrive/DS8_SIA_Project'
BRANCH = 'dashboard'

%cd {REPO_PATH}
!git checkout {BRANCH}
!git pull origin {BRANCH}
print('✅ 준비 완료')

## 3. 패키지 설치

In [ ]:
!pip install skyfield geopy -q
print('✅ 패키지 설치 완료')

## 4. 위성 스케줄링 실행

In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
from skyfield.api import EarthSatellite, load, wgs84
from geopy.distance import geodesic

# ── 데이터 로드 ───────────────────────────────────────────
with open('project/01_data/raw/tle_eo_sar.json') as f:
    tle_data = json.load(f)

df_events = pd.read_csv('project/01_data/processed/final_priority_geo.csv')
df_events['SQLDATE'] = pd.to_datetime(df_events['SQLDATE'])

# ── 중복 좌표 제거 ────────────────────────────────────────
df_events = df_events.drop_duplicates(subset=['ActionGeo_Lat', 'ActionGeo_Long', 'SQLDATE'])
df_events = df_events.reset_index(drop=True)

print(f'대상 이벤트 수 (중복 제거 후): {len(df_events)}개')
print(f'대상 위성 수: {len(tle_data)}개')

In [ ]:
# ── 위성 객체 생성 ────────────────────────────────────────
ts = load.timescale()

satellites = []
for sat_data in tle_data:
    try:
        sat = EarthSatellite.from_omm(ts, sat_data)
        satellites.append({
            'name': sat_data.get('OBJECT_NAME', 'UNKNOWN'),
            'norad_id': sat_data.get('NORAD_CAT_ID'),
            'satellite': sat
        })
    except:
        continue

print(f'위성 객체 생성: {len(satellites)}개')

In [ ]:
# ── 궤도 계산 ─────────────────────────────────────────────
PROXIMITY_KM = 30  # 반경 30km
results = []

for idx, event in df_events.iterrows():
    event_lat  = event['ActionGeo_Lat']
    event_lon  = event['ActionGeo_Long']
    event_date = event['SQLDATE']

    t_start = ts.from_datetime(
        event_date.to_pydatetime().replace(tzinfo=timezone.utc) - timedelta(hours=12)
    )
    t_end = ts.from_datetime(
        event_date.to_pydatetime().replace(tzinfo=timezone.utc) + timedelta(hours=12)
    )
    times = ts.linspace(t_start, t_end, 144)

    for sat_info in satellites:
        sat = sat_info['satellite']
        try:
            geocentric = sat.at(times)
            subpoint   = wgs84.subpoint_of(geocentric)
            lats = subpoint.latitude.degrees
            lons = subpoint.longitude.degrees

            min_dist = min(
                geodesic((event_lat, event_lon), (lat, lon)).km
                for lat, lon in zip(lats, lons)
            )

            if min_dist <= PROXIMITY_KM:
                track_lats = [round(float(lat), 4) for lat in lats]
                track_lons = [round(float(lon), 4) for lon in lons]

                results.append({
                    'SQLDATE':        event['SQLDATE'],
                    'event_lat':      event_lat,
                    'event_lon':      event_lon,
                    'priority_score': event['priority_score'],
                    'satellite_name': sat_info['name'],
                    'norad_id':       sat_info['norad_id'],
                    'min_dist_km':    round(min_dist, 1),
                    'track_lats':     str(track_lats),
                    'track_lons':     str(track_lons)
                })
        except:
            continue

    if (idx + 1) % 50 == 0:
        print(f'진행: {idx+1}/{len(df_events)} 이벤트 처리 완료 | 탐지: {len(results)}건')

print(f'\n✅ 완료 | 근접 궤도 탐지: {len(results)}건')

## 5. 저장 및 GitHub Push

In [ ]:
# ── CSV 저장 ──────────────────────────────────────────────
df_results = pd.DataFrame(results)
df_results.to_csv('project/01_data/processed/satellite_passes.csv', index=False)
print(f'✅ 저장 완료: satellite_passes.csv ({len(df_results)}행)')

In [ ]:
# ── GitHub Push ───────────────────────────────────────────
# GitHub 사용자 정보 설정 (본인 정보로 수정)
GIT_EMAIL = 'your-email@gmail.com'  # 본인 GitHub 이메일
GIT_NAME  = 'Your Name'             # 본인 이름

!git config user.email '{GIT_EMAIL}'
!git config user.name '{GIT_NAME}'
!git add project/01_data/processed/satellite_passes.csv
!git commit -m 'update satellite_passes.csv (30km, dedup)'
!git push origin dashboard
print('✅ GitHub Push 완료')